# 4. Differentiate and fit through equilibrium

**Goal:** replace lesson 3's fixed pose with a solved displacement $u^*(p)$.
Keep the same Neo-Hookean material and mesh. Only the observation model changes.

Given a fixed load, native Newton finds the free displacement satisfying
$g_f(u,p)=\partial\Pi/\partial u_f=0$. We fit the two material parameters from
synthetic free-displacement observations.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, solver
from litepgo.torch import TorchVolumetricMaterial, response
from common import make_geometry

torch.set_num_threads(1)
dtype = torch.float64

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

X, topology, mapping, geometry, plastic = make_geometry()
key = (TorchNeoHookean, fem.TetP1Basis, fem.TetCentroidQuadrature)
truth = np.array([2., 3.])

## Boundary conditions and a fixed load

Fix four vertices and allow vertex 3's three displacement components to move.
The external potential is $-f^Tu$; a `QuadraticEnergy` with a zero quadratic matrix
expresses exactly this linear term. The load and constraints are parameter-independent.

In [ ]:
free = np.array([9, 10, 11])
fixed = dof.DofReduction(mapping.num_dofs, [*range(9), 12, 13, 14])
initial_u = np.zeros(mapping.num_dofs)
load = np.zeros(mapping.num_dofs)
load[free] = [0.015, -0.01, 0.02]
import scipy.sparse as sp
external = energy.QuadraticEnergy(sp.csc_matrix((mapping.num_dofs, mapping.num_dofs)), -load)
options = solver.NewtonOptions(gradient_tolerance=1e-12, max_iterations=100)

## The two factories must describe the same system

`make_energy(p)` returns total potential at parameters `p`.
`make_response(u)` returns the internal energy gradient as a parameter response at
fixed `u`. Its parameter derivative also equals the total-gradient derivative because
the external load is fixed. Both factories use the same material and parameter mapping.

These functions are deliberately visible here: they are the interface between your
physics and the equilibrium differentiator.

In [ ]:
parameter_map = dof.ConstantDofMap(topology, 2)

def make_energy(p):
    fields = [energy.AffineResponse(np.zeros((2, 3)), p)] * topology.num_cells
    elastic = fem.VolumetricFemEnergy[key](mapping, geometry, plastic, fields)
    return energy.WeightedSumEnergy([(elastic, 1.), (external, 1.)])

def make_response(u):
    return fem.VolumetricFemForceMaterialResponse[key](parameter_map, mapping, geometry, u, plastic)

equilibrium = solver.StaticEquilibriumResponse(2, fixed, initial_u, options, make_energy, make_response)

## Define the same kind of outer loss

The reference parameters generate target displacements. The scale is fixed to the
initial RMS displacement discrepancy, making the initial loss exactly 0.5. It is
not recomputed inside the loss.

In [ ]:
target = torch.from_numpy(equilibrium.eval_value(truth).copy())
q = torch.tensor(np.log([1.2, 2.0]), dtype=dtype, requires_grad=True)
start = torch.from_numpy(equilibrium.eval_value(q.detach().exp().numpy()).copy())
scale = (start[free] - target[free]).square().mean().sqrt().item()
assert scale > 1e-10

def loss(q):
    predicted = response(equilibrium, q.exp())
    return 0.5 * ((predicted[free] - target[free]) / scale).square().mean()

print('target free displacement:', target[free].numpy())
print('initial loss:', loss(q).item())

## What does backward calculate?

Let $K_{ff}=\partial g_f/\partial u_f$ and $B_f=\partial g_f/\partial p$.
Differentiating equilibrium gives $K_{ff}\,du_f^*/dp=-B_f$.
For a scalar loss, the adjoint method solves

$$K_{ff}^T a=\partial L/\partial u_f,\qquad dL/dp=-B_f^Ta.$$

`StaticEquilibriumResponse` performs these native solves using the exact unshifted
stiffness. The Torch bridge calls its VJP. It does **not** backpropagate through
Newton iterations; currently backward re-solves equilibrium.

The finite-difference check below also re-solves equilibrium for every perturbation.
It checks the total derivative, unlike the fixed-pose partial derivative in lesson 3.

In [ ]:
analytic = torch.autograd.grad(loss(q), q)[0]
h = 1e-4
numerical = torch.stack([(loss(q.detach() + d) - loss(q.detach() - d)) / (2 * h)
                         for d in h * torch.eye(len(q), dtype=dtype)])
torch.testing.assert_close(analytic, numerical, rtol=2e-4, atol=2e-6)
print('analytic gradient:', analytic.detach().numpy())
print('finite difference:', numerical.detach().numpy())

## Fit and check physical equilibrium

The observation is now displacement, but the optimizer loop is unchanged. Weight or
parameter uniqueness is a separate question; this tutorial requires a good observation fit.

In [ ]:
history = []
optimizer = torch.optim.LBFGS([q], max_iter=25, tolerance_grad=1e-10,
                            tolerance_change=1e-14, line_search_fn='strong_wolfe')

def closure():
    optimizer.zero_grad()
    value = loss(q)
    value.backward()
    history.append(value.item())
    return value

before = loss(q).item()
optimizer.step(closure)
after = loss(q).item()
assert np.isfinite(after) and after < before * 1e-5
print('loss:', before, '->', after)
print('fitted material parameters:', q.detach().exp().numpy())
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.semilogy(np.maximum(history, 1e-18), marker='.')
ax.set(xlabel='L-BFGS closure evaluation (includes line-search trials)', ylabel='Loss')
ax.grid(alpha=0.25)
plt.show()
fitted = q.detach().exp().numpy()
state = equilibrium.eval_value(fitted)
residual = make_energy(fitted).eval_gradient(state)[free]
assert np.linalg.norm(residual) < 1e-9
print('free equilibrium residual norm:', np.linalg.norm(residual))

## Try it

Change the load direction and regenerate the target. Then fit using only one component
of displacement. What happens to parameter recovery even if loss becomes small?

**Checkpoint:** you can distinguish a material partial derivative from a total derivative
through equilibrium. Next: replace the density with a small neural correction.